# Microgrid - In-Class Example 3 (sizing / planning)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)


In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    Binary, NonNegativeReals, Any, minimize, value
)

# ---- Data: loaded from external file 'MG_IC_e3_data.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('MG_IC_e3_data.txt')

GEN_data     = _d['GEN']
PERIOD_data  = _d['PERIOD']
STO_data     = _d['STORAGE']
gen_type     = _d['gen_type']
genMinCap    = _d['genMinCap']
genMaxCap    = _d['genMaxCap']
genInitCost  = _d['genInitCost']
genOpCost    = _d['genOpCost']
genCapFactor = _d['genCapFactor']
ES_MinP      = _d['ES_MinP']
ES_MaxP      = _d['ES_MaxP']
ES_Duration  = _d['ES_Duration']
ES_InitCost  = _d['ES_InitCost']
ES_SOC_Min   = _d['ES_SOC_Min']
ES_SOC_Max   = _d['ES_SOC_Max']
ES_EffiC     = _d['ES_EffiC']
ES_EffiD     = _d['ES_EffiD']
Time_TotalPd = _d['Time_TotalPd']
GridPrice    = _d['GridPrice']

TL_Limit = 2500
MG_Year  = 10
bigM     = 10e9
m = ConcreteModel()
m.GEN     = Set(initialize=GEN_data,    ordered=True)
m.PERIOD  = Set(initialize=PERIOD_data, ordered=True)
m.STORAGE = Set(initialize=STO_data,    ordered=True)

# Generator data
m.gen_type     = Param(m.GEN, initialize=gen_type, within=Any)
m.genMinCap    = Param(m.GEN, initialize=genMinCap)
m.genMaxCap    = Param(m.GEN, initialize=genMaxCap)
m.genInitCost  = Param(m.GEN, initialize=genInitCost)
m.genOpCost    = Param(m.GEN, initialize=genOpCost)
m.genCapFactor = Param(m.GEN, initialize=genCapFactor)

# Storage data
m.ES_MinP     = Param(m.STORAGE, initialize=ES_MinP)
m.ES_MaxP     = Param(m.STORAGE, initialize=ES_MaxP)
m.ES_Duration = Param(m.STORAGE, initialize=ES_Duration)
m.ES_InitCost = Param(m.STORAGE, initialize=ES_InitCost)
m.ES_SOC_Min  = Param(m.STORAGE, initialize=ES_SOC_Min)
m.ES_SOC_Max  = Param(m.STORAGE, initialize=ES_SOC_Max)
m.ES_EffiC    = Param(m.STORAGE, initialize=ES_EffiC)
m.ES_EffiD    = Param(m.STORAGE, initialize=ES_EffiD)

m.Time_TotalPd = Param(m.PERIOD, initialize=Time_TotalPd)
m.GridPrice    = Param(m.PERIOD, initialize=GridPrice)

# Variables
m.uBuild  = Var(m.GEN, domain=Binary)
m.PgSize  = Var(m.GEN, domain=NonNegativeReals)
m.Pg      = Var(m.GEN, m.PERIOD, domain=NonNegativeReals)

m.uBuild_ES = Var(m.STORAGE, domain=Binary)
m.P_Size_ES = Var(m.STORAGE, domain=NonNegativeReals)
m.ES_EInit  = Var(m.STORAGE, domain=NonNegativeReals)
m.u_c_ES    = Var(m.STORAGE, m.PERIOD, domain=Binary)
m.u_d_ES    = Var(m.STORAGE, m.PERIOD, domain=Binary)
m.P_c_ES    = Var(m.STORAGE, m.PERIOD, domain=NonNegativeReals)
m.P_d_ES    = Var(m.STORAGE, m.PERIOD, domain=NonNegativeReals)
m.E_ES      = Var(m.STORAGE, m.PERIOD, domain=NonNegativeReals)

m.Pgrid     = Var(m.PERIOD, bounds=(-TL_Limit, TL_Limit))

# Objective: capital + lifetime operating + lifetime grid-import cost (per AMPL formulation)
m.obj = Objective(
    rule=lambda mm:
        sum(mm.genInitCost[g]*mm.PgSize[g] for g in mm.GEN)
      + sum(mm.ES_InitCost[e]*mm.P_Size_ES[e] for e in mm.STORAGE)
      + MG_Year*365*sum(mm.genOpCost[g]*mm.Pg[g,t]*8 for g in mm.GEN for t in mm.PERIOD)
      + MG_Year*365*sum(mm.GridPrice[t]*mm.Pgrid[t]*8 for t in mm.PERIOD),
    sense=minimize
)

# Constraints
def pb_rule(mm, t):
    return sum(mm.Pg[g,t] for g in mm.GEN) == mm.Time_TotalPd[t] - mm.Pgrid[t] \
           + sum(mm.P_c_ES[e,t] - mm.P_d_ES[e,t] for e in mm.STORAGE)
m.PowerBalance = Constraint(m.PERIOD, rule=pb_rule)

m.genLimit_Max = Constraint(m.GEN, m.PERIOD,
    rule=lambda mm,g,t: mm.Pg[g,t] <= mm.PgSize[g]*mm.genCapFactor[g])
m.gen_Sizing1  = Constraint(m.GEN, rule=lambda mm,g: mm.genMinCap[g]*mm.uBuild[g] <= mm.PgSize[g])
m.gen_Sizing2  = Constraint(m.GEN, rule=lambda mm,g: mm.PgSize[g] <= mm.genMaxCap[g]*mm.uBuild[g])

def solar_rule(mm, g, t):
    if mm.gen_type[g] == 'Solar' and t > mm.PERIOD.first():
        return mm.Pg[g,t] == 0
    return Constraint.Skip
m.solarLimit = Constraint(m.GEN, m.PERIOD, rule=solar_rule)

m.ES_Sizing1 = Constraint(m.STORAGE, rule=lambda mm,e: mm.uBuild_ES[e]*mm.ES_MinP[e] <= mm.P_Size_ES[e])
m.ES_Sizing2 = Constraint(m.STORAGE, rule=lambda mm,e: mm.P_Size_ES[e] <= mm.uBuild_ES[e]*mm.ES_MaxP[e])

m.ESLimit_EMin = Constraint(m.STORAGE, m.PERIOD,
    rule=lambda mm,e,t: mm.ES_SOC_Min[e]*mm.P_Size_ES[e]*mm.ES_Duration[e] <= mm.E_ES[e,t])
m.ESLimit_EMax = Constraint(m.STORAGE, m.PERIOD,
    rule=lambda mm,e,t: mm.E_ES[e,t] <= mm.ES_SOC_Max[e]*mm.P_Size_ES[e]*mm.ES_Duration[e])

m.ESLimit_Mode   = Constraint(m.STORAGE, m.PERIOD,
    rule=lambda mm,e,t: mm.u_c_ES[e,t] + mm.u_d_ES[e,t] <= mm.uBuild_ES[e])
m.ESLimit_PMaxC  = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.P_c_ES[e,t] <= bigM*mm.u_c_ES[e,t])
m.ESLimit_PMaxC2 = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.P_c_ES[e,t] <= mm.P_Size_ES[e])
m.ESLimit_PMaxD  = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.P_d_ES[e,t] <= bigM*mm.u_d_ES[e,t])
m.ESLimit_PMaxD2 = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.P_d_ES[e,t] <= mm.P_Size_ES[e])

def E_calc(mm,e,t):
    if t == mm.PERIOD.first():
        return mm.E_ES[e,t] == mm.ES_EInit[e] + 8*(mm.ES_EffiC[e]*mm.P_c_ES[e,t] - mm.P_d_ES[e,t]/mm.ES_EffiD[e])
    return mm.E_ES[e,t] == mm.E_ES[e, mm.PERIOD.prev(t)] + 8*(mm.ES_EffiC[e]*mm.P_c_ES[e,t] - mm.P_d_ES[e,t]/mm.ES_EffiD[e])
m.ES_Ecalc = Constraint(m.STORAGE, m.PERIOD, rule=E_calc)
m.ES_ESame = Constraint(m.STORAGE, rule=lambda mm,e: mm.E_ES[e, mm.PERIOD.last()] == mm.ES_EInit[e])

# Off-grid resilience requirements
m.OffGrid_Req1 = Constraint(
    expr=sum(m.PgSize[g]*m.genCapFactor[g] for g in m.GEN) >= m.Time_TotalPd[m.PERIOD.first()]
)
def offgrid2(mm, t):
    if t == mm.PERIOD.first(): return Constraint.Skip
    return sum(mm.PgSize[g]*mm.genCapFactor[g] for g in mm.GEN if mm.gen_type[g] != 'Solar') >= mm.Time_TotalPd[t]
m.OffGrid_Req2 = Constraint(m.PERIOD, rule=offgrid2)

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)

m = model

def v(x): return int(round(value(x)))   # for binaries
def f(x): return f"{value(x):.4g}"      # general number

# ---- AMPL-style output ----
print(":   u_c_ES u_d_ES    :=")
for e in m.STORAGE:
    for t in m.PERIOD:
        print(f"{e} {t}    {v(m.u_c_ES[e,t])}      {v(m.u_d_ES[e,t])}")
print(";\n")

print(":   P_c_ES P_d_ES    :=")
for e in m.STORAGE:
    for t in m.PERIOD:
        print(f"{e} {t}    {f(m.P_c_ES[e,t])}      {f(m.P_d_ES[e,t])}")
print(";\n")

print("E_ES :=")
for e in m.STORAGE:
    for t in m.PERIOD:
        print(f"{e} {t}   {f(m.E_ES[e,t])}")
print(";\n")

print("Pg :=")
for g in m.GEN:
    for t in m.PERIOD:
        print(f"{g} {t}   {f(m.Pg[g,t])}")
print(";\n")

print("Pgrid [*] :=")
for t in m.PERIOD:
    print(f"{t}  {f(m.Pgrid[t])}")
print(";\n")

print(": uBuild PgSize    :=")
for g in m.GEN:
    print(f"{g}    {v(m.uBuild[g])}     {f(m.PgSize[g])}")
print(";\n")

print(": uBuild_ES P_Size_ES    :=")
for e in m.STORAGE:
    print(f"{e}      {v(m.uBuild_ES[e])}        {f(m.P_Size_ES[e])}")
print(";\n")

print(f"Lifetime objective = {value(m.obj):.0f}")


Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmp31d2j7mq.pyomo.lp


Reading time = 0.00 seconds
x1: 50 rows, 36 columns, 120 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]


Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90
MIPGap  0



Optimize a model with 50 rows, 36 columns and 120 nonzeros


Model fingerprint: 0x5f49f6a8


...

H    0     0                    -711600.0000 -711600.00  0.00%     -    0s
     0     0 -711600.00    0    4 -711600.00 -711600.00  0.00%     -    0s

Explored 1 nodes (17 simplex iterations) in 0.01 seconds (0.00 work units)


Thread count was 20 (of 20 available processors)



Solution count 2: -711600 -90000 
No other solutions better than -711600

Optimal solution found (tolerance 0.00e+00)


Best objective -7.116000000000e+05, best bound -7.116000000000e+05, gap 0.0000%


ok optimal
:   u_c_ES u_d_ES    :=
1 1    1      0
1 2    0      1
1 3    1      0
;

:   P_c_ES P_d_ES    :=
1 1    0      0
1 2    0      240
1 3    240      0
;

E_ES :=
1 1   2160
1 2   240
1 3   2160
;

Pg :=
1 1   1000
1 2   0
1 3   0
2 1   1500
2 2   1500
2 3   0
3 1   1760
3 2   1760
3 3   0
;

Pgrid [*] :=
1  -1760
2  -2500
3  1240
;

: uBuild PgSize    :=
1    1     4000
2    1     1500
3    1     1760
;

: uBuild_ES P_Size_ES    :=
1      1        600
;

Lifetime objective = -711600
